# 1.1.0 Dimensionality reduction

The reduced embedding spaces used in the downstream analysis are selected from a grid over reducer family, dimensionality, and layer aggregation using three criteria: `knn_recall`, `trustworthiness`, and `drift_correlation`. For the inferential notebooks, the current default remains `PCA-100_all_mean`.

Load mention-level embeddings for the main analysis corpus, evaluate several dimensionality-reduction methods, and save the strongest reduced-space artifacts for downstream notebooks.

---

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def _find_workspace_root() -> Path:
    env_root = os.environ.get("WORKSPACE_ROOT", "").strip()
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "config" / "workspace.json").exists():
            return candidate
    for start in [Path.cwd(), *Path.cwd().parents]:
        if (start / "config" / "workspace.json").exists():
            return start
    raise FileNotFoundError(
        "Could not locate workspace root from WORKSPACE_ROOT or the current working directory."
    )


WORKSPACE_DIR = _find_workspace_root()
PAPER_ROOT = WORKSPACE_DIR.parent
RESEARCH_ROOT = PAPER_ROOT.parent
SHARED_ASSETS_DIR = RESEARCH_ROOT / "shared-assets"
SHARED_CODE_DIR = SHARED_ASSETS_DIR / "code"
if str(SHARED_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(SHARED_CODE_DIR))

CODE_DIR = WORKSPACE_DIR / "code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from workspace_rooting.workspace_paths import canonical_workspace_paths

paths = canonical_workspace_paths(WORKSPACE_DIR)
WORKSPACE_DIR = paths["workspace"]
CODE_DIR = paths["code"]
CONFIG_DIR = paths["config"]
DATA_DIR = paths["data"]
OUTPUTS_DIR = paths["outputs"]
DOCS_DIR = paths["docs"]
LOCAL_DIR = paths["local"]
SHARED_ASSETS_DIR = paths["shared_assets"]

print("workspace_root:", WORKSPACE_DIR)
print("shared_assets_root:", SHARED_ASSETS_DIR)

import hashlib
import json
from pathlib import Path

import joblib
import matplotlib as mpl
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import umap
from IPython import get_ipython
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.manifold import trustworthiness
from sklearn.neighbors import NearestNeighbors
from sklearn.random_projection import GaussianRandomProjection

mpl.rcParams["font.family"] = "Stix Two Math"
ip = get_ipython()
if ip is not None:
    ip.run_line_magic("config", 'InlineBackend.figure_format = "retina"')

RUN_ID = "analysis_corpus"
AVAILABLE_LAYER_MODES = ["last1", "last4_mean", "mid4_mean", "all_mean", "weighted"]
BEST_LAYER_MODE = "all_mean"
LAYER_MODE_POLICY = "best_only"  # "best_only" | "all"
EXPLICIT_LAYER_MODES = None  # e.g. ["all_mean"] or ["mid4_mean", "all_mean"]; overrides LAYER_MODE_POLICY
AVAILABLE_DIMS = [25, 50, 75, 100, 200]
BEST_DIM = 200
DIM_POLICY = "best_only"  # "best_only" | "all"
EXPLICIT_DIMS = None  # e.g. [100] or [100, 200]; overrides DIM_POLICY
K_NEIGHBORS = 15
RANDOM_STATE = 42
RUN_MODE = "resume"  # "resume" | "replay"


def resolve_layer_modes(
    available_layer_modes: list[str],
    best_layer_mode: str,
    layer_mode_policy: str,
    explicit_layer_modes: list[str] | None,
) -> list[str]:
    if explicit_layer_modes is not None:
        unknown = [mode for mode in explicit_layer_modes if mode not in available_layer_modes]
        if unknown:
            raise ValueError(f"Unknown explicit layer modes: {unknown}")
        resolved = explicit_layer_modes
    elif layer_mode_policy == "all":
        resolved = available_layer_modes
    elif layer_mode_policy == "best_only":
        if best_layer_mode not in available_layer_modes:
            raise ValueError(f"BEST_LAYER_MODE not found in AVAILABLE_LAYER_MODES: {best_layer_mode}")
        resolved = [best_layer_mode]
    else:
        raise ValueError(f"Unknown LAYER_MODE_POLICY: {layer_mode_policy}")

    if not resolved:
        raise ValueError("Resolved layer mode list is empty.")

    return resolved


def resolve_dims(
    available_dims: list[int],
    best_dim: int,
    dim_policy: str,
    explicit_dims: list[int] | None,
) -> list[int]:
    if explicit_dims is not None:
        unknown = [dim for dim in explicit_dims if dim not in available_dims]
        if unknown:
            raise ValueError(f"Unknown explicit dims: {unknown}")
        resolved = explicit_dims
    elif dim_policy == "all":
        resolved = available_dims
    elif dim_policy == "best_only":
        if best_dim not in available_dims:
            raise ValueError(f"BEST_DIM not found in AVAILABLE_DIMS: {best_dim}")
        resolved = [best_dim]
    else:
        raise ValueError(f"Unknown DIM_POLICY: {dim_policy}")

    if not resolved:
        raise ValueError("Resolved dimension list is empty.")

    return resolved


LAYER_MODES = resolve_layer_modes(
    available_layer_modes=AVAILABLE_LAYER_MODES,
    best_layer_mode=BEST_LAYER_MODE,
    layer_mode_policy=LAYER_MODE_POLICY,
    explicit_layer_modes=EXPLICIT_LAYER_MODES,
)
DIMS = resolve_dims(
    available_dims=AVAILABLE_DIMS,
    best_dim=BEST_DIM,
    dim_policy=DIM_POLICY,
    explicit_dims=EXPLICIT_DIMS,
)

EVAL_KNN_RECALL = True
EVAL_TRUSTWORTHINESS = True
EVAL_DRIFT_CORRELATION = True
METRIC_TOGGLES = {
    "knn_recall": EVAL_KNN_RECALL,
    "trustworthiness": EVAL_TRUSTWORTHINESS,
    "drift_correlation": EVAL_DRIFT_CORRELATION,
}
ACTIVE_METRICS = [name for name, enabled in METRIC_TOGGLES.items() if enabled]
if not ACTIVE_METRICS:
    raise ValueError("Enable at least one evaluation metric.")

AVAILABLE_REDUCERS = ["pca", "grp", "umap", "ae", "vae"]
BEST_REDUCER = "ae"
REDUCER_POLICY = "best_only"  # "best_only" | "all"
EXPLICIT_REDUCERS = None  # e.g. ["ae"] to run a custom subset explicitly


def resolve_reducers(
    available_reducers: list[str],
    best_reducer: str,
    reducer_policy: str,
    explicit_reducers: list[str] | None,
) -> list[str]:
    if explicit_reducers is not None:
        unknown = [name for name in explicit_reducers if name not in available_reducers]
        if unknown:
            raise ValueError(f"Unknown explicit reducers: {unknown}")
        resolved = explicit_reducers
    elif reducer_policy == "all":
        resolved = available_reducers
    elif reducer_policy == "best_only":
        if best_reducer not in available_reducers:
            raise ValueError(f"BEST_REDUCER not found in AVAILABLE_REDUCERS: {best_reducer}")
        resolved = [best_reducer]
    else:
        raise ValueError(f"Unknown REDUCER_POLICY: {reducer_policy}")

    if not resolved:
        raise ValueError("Resolved reducer list is empty.")

    return resolved


ACTIVE_REDUCERS = resolve_reducers(
    available_reducers=AVAILABLE_REDUCERS,
    best_reducer=BEST_REDUCER,
    reducer_policy=REDUCER_POLICY,
    explicit_reducers=EXPLICIT_REDUCERS,
)

RUN_PCA = "pca" in ACTIVE_REDUCERS
RUN_GRP = "grp" in ACTIVE_REDUCERS
RUN_UMAP = "umap" in ACTIVE_REDUCERS
RUN_AE = "ae" in ACTIVE_REDUCERS
RUN_VAE = "vae" in ACTIVE_REDUCERS

CORPUS_ROOT = LOCAL_DIR / "corpora" / RUN_ID
CORPUS_DIR = CORPUS_ROOT / "bins"
CORPUS_MANIFEST = CORPUS_ROOT / "corpus_manifest.json"
RUN_ROOT = OUTPUTS_DIR / "analytical-results" / RUN_ID
EMB_ROOT = RUN_ROOT / "embeddings"
OUT_DIR = OUTPUTS_DIR / "dimension-reduction" / "models" / RUN_ID
EVAL_DIR = OUT_DIR / "evaluation"
CHECKPOINT = EVAL_DIR / "dim_reduction_eval.json"
CHECKPOINT_META_KEY = "_cache_meta"

EVAL_DIR.mkdir(parents=True, exist_ok=True)
for reducer_name, enabled in {
    "pca": RUN_PCA,
    "grp": RUN_GRP,
    "umap": RUN_UMAP,
    "ae": RUN_AE,
    "vae": RUN_VAE,
}.items():
    if enabled:
        (OUT_DIR / reducer_name).mkdir(parents=True, exist_ok=True)

print("CORPUS_DIR:", CORPUS_DIR, "exists:", CORPUS_DIR.exists())
print("EMB_ROOT:", EMB_ROOT, "exists:", EMB_ROOT.exists())
print("OUT_DIR:", OUT_DIR)
print("RUN_MODE:", RUN_MODE)
print("LAYER_MODES:", LAYER_MODES)
print("DIMS:", DIMS)
print("ACTIVE_REDUCERS:", ACTIVE_REDUCERS)


## Helper functions
 For loading corpus assets, validating embeddings, and evaluating reductions

---

In [ ]:
def bin_start_year(label: str) -> int:
    return int(str(label).split("-")[0])

def example_key(row: dict) -> str:
    sent = " ".join(str(row["sent"]).split())
    return f'{row["bibcode"]}|||{sent}|||{int(row["start"])}|||{int(row["end"])}'

def load_corpus_manifest(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing corpus manifest: {path}")
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    required = ["written_bins", "final_bin_counts", "bin_content_hashes"]
    missing = [key for key in required if key not in data]
    if missing:
        raise ValueError(f"Corpus manifest missing required keys: {missing}")
    return data

def load_corpus_df(corpus_dir: Path) -> tuple[pd.DataFrame, list[str]]:
    bins = sorted([p.stem for p in corpus_dir.glob("*.jsonl")], key=bin_start_year)
    if not bins:
        raise FileNotFoundError(f"No .jsonl bins found in {corpus_dir}")

    rows = []
    for bin_label in bins:
        with open(corpus_dir / f"{bin_label}.jsonl", encoding="utf-8") as f:
            for line in f:
                row = json.loads(line)
                row["bin"] = bin_label
                row["example_key"] = example_key(row)
                rows.append(row)

    return pd.DataFrame(rows), bins

def save_checkpoint(results: dict, cache_meta: dict) -> None:
    payload = {CHECKPOINT_META_KEY: cache_meta, **results}
    with open(CHECKPOINT, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)
    print(f"  ✓ checkpoint saved -> {CHECKPOINT}")

def load_checkpoint() -> tuple[dict, dict]:
    if not CHECKPOINT.exists():
        print("No checkpoint found; starting fresh.")
        return {}, {}

    with open(CHECKPOINT, encoding="utf-8") as f:
        data = json.load(f)

    cache_meta = data.get(CHECKPOINT_META_KEY, {})
    results = {k: v for k, v in data.items() if k != CHECKPOINT_META_KEY}
    print(f"Loaded checkpoint with {len(results)} completed entries")
    return results, cache_meta

def reduction_artifact_paths(kind: str, dim: int, layer_mode: str) -> dict:
    base = OUT_DIR / kind
    paths = {
        "array": base / f"{kind}_{dim}d_{layer_mode}.npy",
        "aux": [],
    }
    if kind in {"pca", "grp"}:
        paths["aux"].append(base / f"{kind}_{dim}_{layer_mode}.pkl")
    elif kind == "ae":
        paths["aux"].append(base / f"autoencoder_{dim}d_{layer_mode}.pt")
    elif kind == "vae":
        paths["aux"].append(base / f"vae_{dim}d_{layer_mode}.pt")
    return paths

def embedding_source_signature(layer_mode: str, bins: list[str], corpus_hashes: dict, corpus_counts: dict) -> str:
    emb_dir = EMB_ROOT / layer_mode
    signature_rows = []

    for bin_label in bins:
        emb_path = emb_dir / f"{bin_label}.pt"
        meta_path = emb_dir / f"{bin_label}.meta.json"
        if not emb_path.exists() or not meta_path.exists():
            raise FileNotFoundError(f"Missing embedding artifact for {layer_mode}/{bin_label}")

        with open(meta_path, "r", encoding="utf-8") as f:
            meta = json.load(f)

        expected_hash = corpus_hashes[bin_label]
        expected_rows = corpus_counts[bin_label]
        cached_hash = meta.get("corpus_bin_sha256")
        cached_rows = meta.get("rows")
        cached_layer_mode = meta.get("layer_mode")

        if cached_layer_mode != layer_mode:
            raise ValueError(
                f"Embedding meta layer mismatch for {layer_mode}/{bin_label}: "
                f"meta has {cached_layer_mode}"
            )
        if cached_rows != expected_rows:
            raise ValueError(
                f"Embedding rows mismatch for {layer_mode}/{bin_label}: "
                f"meta rows={cached_rows}, corpus rows={expected_rows}"
            )
        if cached_hash != expected_hash:
            raise ValueError(
                f"Embedding hash mismatch for {layer_mode}/{bin_label}: "
                f"meta hash={cached_hash}, corpus hash={expected_hash}"
            )

        signature_rows.append({
            "bin": bin_label,
            "rows": cached_rows,
            "corpus_bin_sha256": cached_hash,
            "embedding_file": emb_path.name,
            "embedding_mtime_ns": emb_path.stat().st_mtime_ns,
            "meta_mtime_ns": meta_path.stat().st_mtime_ns,
        })

    payload = {
        "run_id": RUN_ID,
        "layer_mode": layer_mode,
        "bins": signature_rows,
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()

def load_embedding_stack(layer_mode: str, bins: list[str], corpus_hashes: dict, corpus_counts: dict) -> tuple[np.ndarray, str]:
    emb_dir = EMB_ROOT / layer_mode
    X_parts = []
    layer_signature = embedding_source_signature(layer_mode, bins, corpus_hashes, corpus_counts)

    print(f"\nChecking corpus vs embeddings for `{layer_mode}`:")
    ok = True

    for bin_label in bins:
        emb_path = emb_dir / f"{bin_label}.pt"
        meta_path = emb_dir / f"{bin_label}.meta.json"
        with open(meta_path, "r", encoding="utf-8") as f:
            meta = json.load(f)

        X_bin = torch.load(emb_path, map_location="cpu")
        if isinstance(X_bin, torch.Tensor):
            X_bin = X_bin.cpu().numpy()

        n_emb = X_bin.shape[0]
        n_records = int((df["bin"] == bin_label).sum())
        n_meta = meta.get("rows")
        match = "✓" if (n_emb == n_records == n_meta) else "✗ MISMATCH"
        print(
            f"  {bin_label}: records={n_records:>6,} embeddings={n_emb:>6,} "
            f"meta={n_meta:>6,} {match}"
        )

        if not (n_emb == n_records == n_meta):
            ok = False

        X_parts.append(X_bin)

    if not ok:
        raise ValueError(f"Embedding alignment failed for layer_mode={layer_mode}")

    X_all = np.concatenate(X_parts, axis=0)
    if X_all.shape[0] != len(df):
        raise ValueError(f"Global mismatch for {layer_mode}: {X_all.shape[0]} embeddings vs {len(df)} rows")

    print(f"  source signature: {layer_signature[:12]}...")
    return X_all, layer_signature

def cache_is_valid(key: str, paths: dict, layer_signature: str, eval_signature: str, results: dict, cache_meta: dict) -> tuple[bool, str]:
    if key not in results:
        return False, "missing checkpoint entry"

    meta = cache_meta.get(key, {})
    if meta.get("source_signature") != layer_signature:
        return False, "source signature changed"
    if meta.get("eval_signature") != eval_signature:
        return False, "evaluation metric set changed"

    missing = [str(p) for p in [paths["array"], *paths["aux"]] if not p.exists()]
    if missing:
        return False, f"missing artifact(s): {missing}"

    return True, "cache valid"


def replay_cache_is_valid(key: str, paths: dict, results: dict) -> tuple[bool, str]:
    if key not in results:
        return False, "missing checkpoint entry"

    missing = [str(p) for p in [paths["array"], *paths["aux"]] if not p.exists()]
    if missing:
        return False, f"missing artifact(s): {missing}"

    return True, "replay cache available"


# ── Evaluation Functions ─────────────────────────────────────────────────────
def knn_recall(X_orig: np.ndarray, X_reduced: np.ndarray, k: int = 15) -> float:
    nn_orig = NearestNeighbors(n_neighbors=k + 1, n_jobs=-1).fit(X_orig)
    nn_reduced = NearestNeighbors(n_neighbors=k + 1, n_jobs=-1).fit(X_reduced)

    _, idx_orig = nn_orig.kneighbors(X_orig)
    _, idx_reduced = nn_reduced.kneighbors(X_reduced)

    idx_orig = idx_orig[:, 1:]
    idx_reduced = idx_reduced[:, 1:]

    return float(np.mean([
        len(set(idx_orig[i]) & set(idx_reduced[i])) / k
        for i in range(len(X_orig))
    ]))

def drift_correlation(X_orig: np.ndarray, X_reduced: np.ndarray, bins_arr: np.ndarray) -> float:
    bin_list = sorted(set(bins_arr), key=bin_start_year)

    centroids_orig = []
    centroids_reduced = []

    for b in bin_list:
        mask = bins_arr == b
        if mask.sum() < 10:
            continue
        centroids_orig.append(X_orig[mask].mean(axis=0))
        centroids_reduced.append(X_reduced[mask].mean(axis=0))

    dists_orig, dists_reduced = [], []
    for i in range(len(centroids_orig) - 1):
        a1, b1 = centroids_orig[i], centroids_orig[i + 1]
        a2, b2 = centroids_reduced[i], centroids_reduced[i + 1]

        dists_orig.append(
            1 - np.dot(a1, b1) / (np.linalg.norm(a1) * np.linalg.norm(b1) + 1e-10)
        )
        dists_reduced.append(
            1 - np.dot(a2, b2) / (np.linalg.norm(a2) * np.linalg.norm(b2) + 1e-10)
        )

    if len(dists_orig) < 2:
        return float("nan")

    return float(spearmanr(dists_orig, dists_reduced).statistic)

def evaluate(X_reduced: np.ndarray, label: str, X_orig: np.ndarray) -> dict:
    print(f"  Evaluating {label}...")

    metrics = {
        "knn_recall": float("nan"),
        "trustworthiness": float("nan"),
        "drift_correlation": float("nan"),
    }
    summary_parts = []

    if EVAL_KNN_RECALL:
        metrics["knn_recall"] = knn_recall(X_orig, X_reduced, k=K_NEIGHBORS)
        summary_parts.append(f"knn_recall={metrics['knn_recall']:.4f}")
    else:
        summary_parts.append("knn_recall=skipped")

    if EVAL_TRUSTWORTHINESS:
        metrics["trustworthiness"] = float(trustworthiness(X_orig, X_reduced, n_neighbors=K_NEIGHBORS))
        summary_parts.append(f"trustworthiness={metrics['trustworthiness']:.4f}")
    else:
        summary_parts.append("trustworthiness=skipped")

    if EVAL_DRIFT_CORRELATION:
        metrics["drift_correlation"] = drift_correlation(X_orig, X_reduced, bins_all)
        summary_parts.append(f"drift_corr={metrics['drift_correlation']:.4f}")
    else:
        summary_parts.append("drift_corr=skipped")

    print(f"  {label}: " + " | ".join(summary_parts))
    return metrics

# ── Models / Algorithms ──────────────────────────────────────────────────────
class Autoencoder(nn.Module):
    def __init__(self, input_dim: int = 768, bottleneck: int = 100):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, bottleneck),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck, 256),
            nn.ReLU(),
            nn.Linear(256, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

def train_autoencoder(X, bottleneck=100, epochs=30, batch_size=512, lr=1e-3):
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    model = Autoencoder(input_dim=X.shape[1], bottleneck=bottleneck).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    dataset = torch.utils.data.TensorDataset(torch.tensor(X, dtype=torch.float32))
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)

    model.train()
    for epoch in range(epochs):
        total = 0.0
        for (xb,) in loader:
            xb = xb.to(device)
            recon, _ = model(xb)
            loss = loss_fn(recon, xb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item()
        if (epoch + 1) % 5 == 0:
            print(f"    epoch {epoch + 1}/{epochs} | loss {total / len(loader):.5f}")

    model.eval()
    Z_list = []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            xb = torch.tensor(X[start:start + batch_size], dtype=torch.float32).to(device)
            _, z = model(xb)
            Z_list.append(z.cpu().numpy())

    return np.concatenate(Z_list, axis=0), model

class VAE(nn.Module):
    def __init__(self, input_dim: int = 768, bottleneck: int = 100):
        super().__init__()
        self.encoder_mu = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, bottleneck),
        )
        self.encoder_logvar = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, bottleneck),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck, 256),
            nn.ReLU(),
            nn.Linear(256, input_dim),
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu = self.encoder_mu(x)
        logvar = self.encoder_logvar(x)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

def vae_loss(recon, x, mu, logvar, beta=1.0):
    recon_loss = nn.functional.mse_loss(recon, x, reduction="sum")
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kl_loss

def train_vae(X, bottleneck=100, epochs=30, batch_size=512, lr=1e-3, beta=1.0):
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    model = VAE(input_dim=X.shape[1], bottleneck=bottleneck).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    dataset = torch.utils.data.TensorDataset(torch.tensor(X, dtype=torch.float32))
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)

    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        for (xb,) in loader:
            xb = xb.to(device)
            recon, mu, logvar = model(xb)
            loss = vae_loss(recon, xb, mu, logvar, beta=beta)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total_loss += loss.item()
        if (epoch + 1) % 5 == 0:
            print(f"    epoch {epoch + 1}/{epochs} | loss {total_loss / len(loader):.5f}")

    model.eval()
    Z_list = []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            xb = torch.tensor(X[start:start + batch_size], dtype=torch.float32).to(device)
            _, mu, _ = model(xb)
            Z_list.append(mu.cpu().numpy())

    return np.concatenate(Z_list, axis=0), model


# Load corpus + manifest

---

In [ ]:
# ── Load ────────────────────────────────────────────────────────────────
corpus_manifest = load_corpus_manifest(CORPUS_MANIFEST)
df, bins = load_corpus_df(CORPUS_DIR)
bins_all = df["bin"].to_numpy()
corpus_counts = corpus_manifest["final_bin_counts"]
corpus_hashes = corpus_manifest["bin_content_hashes"]

missing_bins = [b for b in bins if b not in corpus_counts or b not in corpus_hashes]
if missing_bins:
    raise ValueError(f"Corpus manifest missing counts or hashes for bins: {missing_bins}")

actual_counts = df["bin"].value_counts().to_dict()
for bin_label in bins:
    if actual_counts.get(bin_label, 0) != corpus_counts[bin_label]:
        raise ValueError(
            f"Corpus/bin mismatch for {bin_label}: dataframe={actual_counts.get(bin_label, 0)} "
            f"manifest={corpus_counts[bin_label]}"
        )

print(df["bin"].value_counts().sort_index())


# Run computations


In [ ]:
# ── Main Loop ────────────────────────────────────────────────────────────────
results, cache_meta = load_checkpoint()
eval_signature = hashlib.sha256(json.dumps(METRIC_TOGGLES, sort_keys=True).encode("utf-8")).hexdigest()
current_run_keys = []

if RUN_MODE not in {"resume", "replay"}:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")

for layer_mode in LAYER_MODES:
    print(f"\n{'#' * 55}\nLAYER MODE = {layer_mode}\n{'#' * 55}")

    if RUN_MODE == "resume":
        X_all_layer, layer_signature = load_embedding_stack(layer_mode, bins, corpus_hashes, corpus_counts)
        print(f"Loaded {X_all_layer.shape} embeddings for {layer_mode}")
    else:
        X_all_layer = None
        layer_signature = None
        print(f"Replaying cached reduction artifacts for {layer_mode}")

    for dim in DIMS:
        print(f"\n{'=' * 55}\nDIM = {dim}\n{'=' * 55}")

        if RUN_PCA:
            key = f"PCA-{dim}_{layer_mode}"
            current_run_keys.append(key)
            paths = reduction_artifact_paths("pca", dim, layer_mode)
            if RUN_MODE == "replay":
                valid, reason = replay_cache_is_valid(key, paths, results)
            else:
                valid, reason = cache_is_valid(key, paths, layer_signature, eval_signature, results, cache_meta)
            if valid:
                print(f"Skipping {key}: {reason}")
            else:
                if RUN_MODE == "replay":
                    raise RuntimeError(f"Replay cache incomplete for {key}: {reason}. Switch RUN_MODE to 'resume'.")
                print(f"Running {key} ({reason})...")
                pca = PCA(n_components=dim, random_state=RANDOM_STATE)
                X_pca = pca.fit_transform(X_all_layer)
                joblib.dump(pca, paths["aux"][0])
                np.save(paths["array"], X_pca)
                results[key] = evaluate(X_pca, key, X_all_layer)
                cache_meta[key] = {
                    "source_signature": layer_signature,
                    "eval_signature": eval_signature,
                    "array_path": str(paths["array"]),
                    "aux_paths": [str(p) for p in paths["aux"]],
                }
                save_checkpoint(results, cache_meta)

        if RUN_GRP:
            key = f"GRP-{dim}_{layer_mode}"
            current_run_keys.append(key)
            paths = reduction_artifact_paths("grp", dim, layer_mode)
            if RUN_MODE == "replay":
                valid, reason = replay_cache_is_valid(key, paths, results)
            else:
                valid, reason = cache_is_valid(key, paths, layer_signature, eval_signature, results, cache_meta)
            if valid:
                print(f"Skipping {key}: {reason}")
            else:
                if RUN_MODE == "replay":
                    raise RuntimeError(f"Replay cache incomplete for {key}: {reason}. Switch RUN_MODE to 'resume'.")
                print(f"Running {key} ({reason})...")
                grp = GaussianRandomProjection(n_components=dim, random_state=RANDOM_STATE)
                X_grp = grp.fit_transform(X_all_layer)
                joblib.dump(grp, paths["aux"][0])
                np.save(paths["array"], X_grp)
                results[key] = evaluate(X_grp, key, X_all_layer)
                cache_meta[key] = {
                    "source_signature": layer_signature,
                    "eval_signature": eval_signature,
                    "array_path": str(paths["array"]),
                    "aux_paths": [str(p) for p in paths["aux"]],
                }
                save_checkpoint(results, cache_meta)

        if RUN_UMAP:
            key = f"UMAP-{dim}_{layer_mode}"
            current_run_keys.append(key)
            paths = reduction_artifact_paths("umap", dim, layer_mode)
            if RUN_MODE == "replay":
                valid, reason = replay_cache_is_valid(key, paths, results)
            else:
                valid, reason = cache_is_valid(key, paths, layer_signature, eval_signature, results, cache_meta)
            if valid:
                print(f"Skipping {key}: {reason}")
            else:
                if RUN_MODE == "replay":
                    raise RuntimeError(f"Replay cache incomplete for {key}: {reason}. Switch RUN_MODE to 'resume'.")
                print(f"Running {key} ({reason})...")
                reducer = umap.UMAP(
                    n_components=dim,
                    n_neighbors=15,
                    min_dist=0.0,
                    random_state=RANDOM_STATE,
                    verbose=True,
                )
                X_umap = reducer.fit_transform(X_all_layer)
                np.save(paths["array"], X_umap)
                results[key] = evaluate(X_umap, key, X_all_layer)
                cache_meta[key] = {
                    "source_signature": layer_signature,
                    "eval_signature": eval_signature,
                    "array_path": str(paths["array"]),
                    "aux_paths": [],
                }
                save_checkpoint(results, cache_meta)

        if RUN_AE:
            key = f"AE-{dim}_{layer_mode}"
            current_run_keys.append(key)
            paths = reduction_artifact_paths("ae", dim, layer_mode)
            if RUN_MODE == "replay":
                valid, reason = replay_cache_is_valid(key, paths, results)
            else:
                valid, reason = cache_is_valid(key, paths, layer_signature, eval_signature, results, cache_meta)
            if valid:
                print(f"Skipping {key}: {reason}")
            else:
                if RUN_MODE == "replay":
                    raise RuntimeError(f"Replay cache incomplete for {key}: {reason}. Switch RUN_MODE to 'resume'.")
                print(f"Running {key} ({reason})...")
                X_ae, ae_model = train_autoencoder(X_all_layer, bottleneck=dim, epochs=30)
                np.save(paths["array"], X_ae)
                torch.save(ae_model.state_dict(), paths["aux"][0])
                results[key] = evaluate(X_ae, key, X_all_layer)
                cache_meta[key] = {
                    "source_signature": layer_signature,
                    "eval_signature": eval_signature,
                    "array_path": str(paths["array"]),
                    "aux_paths": [str(p) for p in paths["aux"]],
                }
                save_checkpoint(results, cache_meta)

        if RUN_VAE:
            key = f"VAE-{dim}_{layer_mode}"
            current_run_keys.append(key)
            paths = reduction_artifact_paths("vae", dim, layer_mode)
            if RUN_MODE == "replay":
                valid, reason = replay_cache_is_valid(key, paths, results)
            else:
                valid, reason = cache_is_valid(key, paths, layer_signature, eval_signature, results, cache_meta)
            if valid:
                print(f"Skipping {key}: {reason}")
            else:
                if RUN_MODE == "replay":
                    raise RuntimeError(f"Replay cache incomplete for {key}: {reason}. Switch RUN_MODE to 'resume'.")
                print(f"Running {key} ({reason})...")
                X_vae, vae_model = train_vae(X_all_layer, bottleneck=dim, epochs=30)
                np.save(paths["array"], X_vae)
                torch.save(vae_model.state_dict(), paths["aux"][0])
                results[key] = evaluate(X_vae, key, X_all_layer)
                cache_meta[key] = {
                    "source_signature": layer_signature,
                    "eval_signature": eval_signature,
                    "array_path": str(paths["array"]),
                    "aux_paths": [str(p) for p in paths["aux"]],
                }
                save_checkpoint(results, cache_meta)

current_run_keys = list(dict.fromkeys(current_run_keys))
current_results = {k: results[k] for k in current_run_keys if k in results}

print(f"\n{'=' * 55}\nCurrent active selection\n{'=' * 55}")
if not current_results:
    print("  No results found for the currently active config.")
else:
    for k, v in sorted(current_results.items()):
        parts = []
        if EVAL_KNN_RECALL:
            parts.append(f"knn_recall={v['knn_recall']:.4f}")
        else:
            parts.append("knn_recall=skipped")
        if EVAL_TRUSTWORTHINESS:
            parts.append(f"trust={v['trustworthiness']:.4f}")
        else:
            parts.append("trust=skipped")
        if EVAL_DRIFT_CORRELATION:
            parts.append(f"drift_corr={v['drift_correlation']:.4f}")
        else:
            parts.append("drift_corr=skipped")
        print(f"  {k:30s} | " + " | ".join(parts))

print(f"\n{'=' * 55}\nAll cached results\n{'=' * 55}")
for k, v in sorted(results.items()):
    parts = []
    if EVAL_KNN_RECALL:
        parts.append(f"knn_recall={v['knn_recall']:.4f}")
    else:
        parts.append("knn_recall=skipped")
    if EVAL_TRUSTWORTHINESS:
        parts.append(f"trust={v['trustworthiness']:.4f}")
    else:
        parts.append("trust=skipped")
    if EVAL_DRIFT_CORRELATION:
        parts.append(f"drift_corr={v['drift_correlation']:.4f}")
    else:
        parts.append("drift_corr=skipped")
    print(f"  {k:30s} | " + " | ".join(parts))


# Results table

---

In [ ]:
# ── Ranked results table ───────────────────────────────────────────────────
results_df = pd.DataFrame.from_dict(results, orient="index")
results_df.index.name = "config"
results_df = results_df.reset_index()

parts = results_df["config"].str.extract(r"^(?P<reducer>[^-]+)-(?P<dim>\d+)_(?P<layer_mode>.+)$")
results_df = pd.concat([results_df, parts], axis=1)
results_df["dim"] = results_df["dim"].astype(int)
results_df["is_current_selection"] = results_df["config"].isin(current_run_keys)

metric_cols = ACTIVE_METRICS.copy()
results_df["overall_score"] = results_df[metric_cols].mean(axis=1)

for col in metric_cols:
    results_df[f"rank_{col}"] = results_df[col].rank(ascending=False, method="min")

results_df["rank_mean"] = results_df[[f"rank_{col}" for col in metric_cols]].mean(axis=1)

results_df = results_df.sort_values(
    by=["overall_score", *metric_cols],
    ascending=[False] * (1 + len(metric_cols)),
).reset_index(drop=True)

ranked_results_path = EVAL_DIR / "dim_reduction_results_ranked.csv"
results_df.to_csv(ranked_results_path, index=False)

current_results_df = results_df.loc[results_df["is_current_selection"]].reset_index(drop=True)
all_cached_results_df = results_df.copy()

print(f"Saved ranked results table -> {ranked_results_path}")
print("\nCurrent active selection:")
print(current_results_df if not current_results_df.empty else "No rows for the current active selection.")
print("\nAll cached results:")
print(all_cached_results_df)
